<a href="https://colab.research.google.com/github/Carleslc/AudioToText/blob/master/AudioToText.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🗣️ AudioToText — WhisperX

**Turn any audio or video into text, subtitles, or a translation — right in your browser.**

Powered by [WhisperX](https://github.com/m-bain/whisperX) (faster Whisper with word-level timestamps). Nothing to install on your computer — it runs free on Google Colab's GPU.

---

### ▶️ How to use

1. **Turn on the GPU:** menu **Runtime → Change runtime type → T4 GPU → Save**.
2. Run each step **in order, top to bottom** — click the round **▶️** button on the left of every cell.
3. Wait for a ✅ before moving to the next step.

> First time here? Just follow Steps 1 → 2 → 3 → 4. Steps 2.5 and 5 are optional extras.

[![Donate](https://www.ko-fi.com/img/githubbutton_sm.svg)](https://ko-fi.com/carleslc)

## 1️⃣ Install (run once)

Click **▶️** below to install everything the notebook needs. This takes a few minutes the first time.

> ⚠️ **The runtime will restart by itself when the install finishes. This is completely normal — don't panic.** Just continue to **Step 2** once it comes back online.

In [ ]:
import os; os.remove("/tmp/deps_installed")

In [ ]:
#@title { display-mode: "form" }
# Faster install: uses `uv` (parallel downloads + fast resolver) instead of pip,
# and skips the ~2.5 GB PyTorch download when Colab already ships torch 2.6/cu124.
# Same 4-phase order as before so the whisperx / ctranslate2 / numpy resolve stays valid.
import os, subprocess, time
from sys import platform as sys_platform

FLAG = "/tmp/deps_installed"

def sh(desc, cmd):
    print(f"\n⏳ {desc}...")
    t = time.time()
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(f"❌ FAILED: {desc}")
        print(r.stdout[-1500:])
        print("--- STDERR ---")
        print(r.stderr[-3000:])
        raise RuntimeError(f"Step failed: {desc}")
    print(f"✅ {desc}  ({time.time() - t:.0f}s)")
    return r

if not os.path.exists(FLAG):

    # ── ffmpeg ────────────────────────────────────────────────────────────────
    if subprocess.run("ffmpeg -version", shell=True, capture_output=True).returncode != 0:
        if sys_platform == 'linux':
            sh("Installing ffmpeg", "apt-get install -y ffmpeg")
        else:
            print("⚠️  Install ffmpeg manually: https://ffmpeg.org/download.html")
    else:
        print("✅ ffmpeg already present")

    # ── uv: fast resolver + parallel downloads (the main speedup) ─────────────
    sh("Installing uv", "pip install -q uv")
    UV = "uv pip install --system"

    # ── Phase 1: PyTorch 2.6.0 (cu124) — skip the big download if already there ─
    need_torch = True
    try:
        import torch
        if torch.__version__.startswith("2.6") and getattr(torch.version, "cuda", "") == "12.4":
            need_torch = False
            print(f"✅ torch {torch.__version__} (cu124) already installed — skipping ~2.5 GB download")
    except ImportError:
        pass
    if need_torch:
        sh("Installing PyTorch 2.6.0 (cu124)",
           f"{UV} torch==2.6.0+cu124 torchaudio==2.6.0+cu124 torchvision==0.21.0+cu124 "
           "--index-url https://download.pytorch.org/whl/cu124")

    # ── Phase 2: pre-pin conflicting deps so the resolver doesn't backtrack ────
    sh("Pre-pinning pyannote/pandas",
       f'{UV} "pandas<3.0.0" "pyannote.core==5.0.0" "pyannote.database==5.1.3"')

    # ── Phase 3: WhisperX + rest. numpy<2.0 forces the compatible whisperx ─────
    sh("Installing WhisperX and dependencies",
       f'{UV} whisperx openai-whisper openai scipy deepl pydub cohere ffmpeg-python '
       '"requests==2.32.4" tensorflow-probability "numpy<2.0"')

    # ── Phase 4: bump ctranslate2 for cuDNN 9 (alone + --no-deps to avoid conflict)
    sh("Upgrading ctranslate2 for cuDNN 9", f'{UV} --no-deps "ctranslate2>=4.5.0"')

    # ── Verify key imports ────────────────────────────────────────────────────
    print("\n🔍 Verifying installs...")
    r = subprocess.run(
        'python -c "import torch, torchaudio, whisperx, numpy, deepl, openai; '
        'print(\'torch\', torch.__version__, \'/ numpy\', numpy.__version__)"',
        shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print("--- STDERR ---")
        print(r.stderr[-3000:])
        raise RuntimeError("One or more packages failed to import. Check errors above.")
    print("  ✅", r.stdout.strip())

    # ── Flag + restart ────────────────────────────────────────────────────────
    open(FLAG, "w").close()
    print("\n✅ Installation complete. Restarting runtime...")
    os.kill(os.getpid(), 9)

else:
    print("✅ Dependencies already installed. Continue to the next step.")

## 2️⃣ Add your audio files

Click **▶️** on the cell below to connect your **Google Drive**.

Then put your audio or video files into this folder (it's created for you automatically):

```
📁 My Drive  ›  for process
```

You can add files two ways:
- **Google Drive** ([drive.google.com](https://drive.google.com)) — open *My Drive → for process* and drag files in.
- **Colab Files panel** — click the 📁 folder icon on the far left, open *drive → MyDrive → for process*, and drag files in.

The cell prints the exact path of each file. **Copy a path** — you'll paste it into `audio_file` in Step 3.

> 💡 Files kept in Drive **stay there between sessions**, so you never have to re-upload.

✅ Almost any audio or video format works ([full list](https://gist.github.com/Carleslc/1d6b922c8bf4a7e9627a6970d178b3a6)).

In [ ]:
#@title 📁 Connect Google Drive { display-mode: "form" }
#@markdown Run this cell, then approve the Google pop-up to connect your Drive.
#@markdown Put your files in the folder below and re-run to see them listed.

#@markdown ---
#@markdown **Folder for your audio/video files** (created automatically if missing):
drive_folder = "/content/drive/MyDrive/for process"  #@param {type:"string"}
#@markdown ---

from google.colab import drive
import os

drive.mount('/content/drive')
os.makedirs(drive_folder, exist_ok=True)

print(f"\n✅ Drive connected. Put your files here: {drive_folder}\n")
print("Files found in that folder (copy a path into Step 3):")
files = sorted(f for f in os.listdir(drive_folder)
               if os.path.isfile(os.path.join(drive_folder, f)))
if files:
    for f in files:
        print("   •", os.path.join(drive_folder, f))
else:
    print("   (empty — add files via the 📁 Files panel on the left, or at drive.google.com)")

## 2️⃣.5 🎙️ Record your own audio *(optional)*

No file to transcribe? Record straight from your microphone instead.

- Run the cell, allow mic access, and a **record button** appears.
- Press it to start, press again to stop.
- Your clip is saved as **`recording.wav`** — use that as the `audio_file` in Step 3.

> Skip this step if you already added a file in Step 2.

In [ ]:
try:
  import io
  import ffmpeg
  import numpy as np

  from google.colab.output import eval_js

  from IPython.display import HTML, Audio
  from scipy.io.wavfile import write, read as wav_read
  from base64 import b64decode
  from os.path import isfile

  AUDIO_HTML = """
  <script>
  var my_div = document.createElement("DIV");
  var my_p = document.createElement("P");
  var my_btn = document.createElement("BUTTON");
  var t = document.createTextNode("Starting recording...");

  my_btn.appendChild(t);
  my_div.appendChild(my_btn);
  document.body.appendChild(my_div);

  var base64data = 0;
  var reader;
  var recorder, gumStream;
  var recordButton = my_btn;

  var handleSuccess = function(stream) {
    gumStream = stream;
    var options = {
      bitsPerSecond: 16000,
      mimeType : 'audio/webm;codecs=opus'
    };
    recorder = new MediaRecorder(stream, options);

    recorder.ondataavailable = function(e) {
      var url = URL.createObjectURL(e.data);
      var preview = document.createElement('audio');
      preview.controls = true;
      preview.src = url;
      document.body.appendChild(preview);

      reader = new FileReader();
      reader.readAsDataURL(e.data);
      reader.onloadend = function() {
        base64data = reader.result;
      }
    };
    recorder.start();
    recordButton.innerText = "🔴 Recording... press to STOP";
  };

  navigator.mediaDevices.getUserMedia({audio: true}).then(handleSuccess);

  function toggleRecording() {
    if (recorder && recorder.state == "recording") {
        recorder.stop();
        gumStream.getAudioTracks()[0].stop();
        recordButton.innerText = "Saving the recording... please wait!";
    }
  }

  function sleep(ms) {
    return new Promise(resolve => setTimeout(resolve, ms));
  }

  var data = new Promise(resolve => {
    recordButton.onclick = () => {
      toggleRecording();
      sleep(2000).then(() => {
        resolve(base64data.toString());
      });
    }
  });

  function doneRecording(recording_file) {
    my_div.removeChild(recordButton);
    my_p.innerText = recording_file;
    my_div.appendChild(my_p);
  }

  </script>
  """

  def get_audio():
    display(HTML(AUDIO_HTML))
    data = eval_js("data")
    binary = b64decode(data.split(',')[1])

    process = (ffmpeg
      .input('pipe:0')
      .output('pipe:1', format='wav')
      .run_async(pipe_stdin=True, pipe_stdout=True, pipe_stderr=True, quiet=True, overwrite_output=True)
    )
    output, err = process.communicate(input=binary)

    riff_chunk_size = len(output) - 8
    q = riff_chunk_size
    b = []
    for i in range(4):
      q, r = divmod(q, 256)
      b.append(r)

    riff = output[:4] + bytes(b) + output[8:]
    sr, audio = wav_read(io.BytesIO(riff))
    return audio, sr

  recording_file = "recording.wav" #@param {type:"string"}

  if isfile(recording_file):
    print(f"{recording_file} already exists, if you want to create another recording with the same name, delete it first")
  else:
    audio, sr = get_audio()
    write(recording_file, sr, audio)
    eval_js(f'doneRecording("{recording_file}")')
except ImportError:
  print("Recording only available in Google Colab")

## 3️⃣ 🎬 Transcribe (or translate)

This is the main step. In the form below:

1. Paste your file path into **`audio_file`** (from Step 2).
2. Pick the **language** and leave the rest on their defaults.
3. Click **▶️** and watch the text appear live.

> Big files take a while — that's normal. When you see **✅ Done**, move on to Step 4 to save.

In [ ]:
#@title 🎬 Step 3 — Transcribe / Translate { display-mode: "form" }
#@markdown Fill in the boxes below, then click **▶️**. Everything runs automatically.
#@markdown The code is hidden to keep things tidy — double-click the title to show it.

import os
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

import subprocess, math, gc
from sys import platform as sys_platform
import logging

from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
print("HF Token loaded ✅")

logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)

import whisperx
from whisper.utils import format_timestamp, get_writer, WriteTXT

import numpy as np

try:
  import tensorflow
except ImportError:
  pass

import torch
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
print("TF32 enabled:", torch.backends.cuda.matmul.allow_tf32)
from openai import OpenAI
import warnings
warnings.filterwarnings("ignore", module="pytorch_lightning")

# ── VRAM cleanup helper ───────────────────────────────────────────────────────
# Frees objects + CUDA cache. Prevents OOM when rerunning this cell for a new
# file: without this, a second whisperx.load_model loads a NEW large-v3 while
# the old one is still resident -> two full models in VRAM -> OOM.
def free_vram(*objs):
  for o in objs:
    try:
      del o
    except Exception:
      pass
  gc.collect()
  if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

# Free a model left in VRAM by a previous run of this cell, before loading again.
try:
  free_vram(model)
  del model
  print("Freed previous model from VRAM.")
except NameError:
  pass
free_vram()

# ── parameters ────────────────────────────────────────────────────────────────

#@markdown ### 📄 What to do
#@markdown **Transcribe** = keep the original language. **Translate to English** = output English.
task = "Transcribe" #@param ["Transcribe", "Translate to English"]
task = "transcribe" if task == "Transcribe" else "translate"

#@markdown ---
#@markdown ### 🎵 Your file
#@markdown Paste the path you copied from Step 2. Do several at once by separating paths with commas.
audio_file = "/content/drive/MyDrive/for process/canan fc 6-29.wav" #@param {type:"string"}
audio_files = [p.strip() for p in audio_file.split(',')]
for audio_path in audio_files:
  if not os.path.isfile(audio_path):
    raise FileNotFoundError(audio_path)

#@markdown ---
#@markdown ### 🌐 Language & quality
#@markdown Spoken language of the audio. Pick **Auto-Detect** if you're not sure.
language = "Japanese" #@param ["Auto-Detect", "Afrikaans", "Albanian", "Amharic", "Arabic", "Armenian", "Assamese", "Azerbaijani", "Bashkir", "Basque", "Belarusian", "Bengali", "Bosnian", "Breton", "Bulgarian", "Burmese", "Castilian", "Catalan", "Chinese", "Croatian", "Czech", "Danish", "Dutch", "English", "Estonian", "Faroese", "Finnish", "Flemish", "French", "Galician", "Georgian", "German", "Greek", "Gujarati", "Haitian", "Haitian Creole", "Hausa", "Hawaiian", "Hebrew", "Hindi", "Hungarian", "Icelandic", "Indonesian", "Italian", "Japanese", "Javanese", "Kannada", "Kazakh", "Khmer", "Korean", "Lao", "Latin", "Latvian", "Letzeburgesch", "Lingala", "Lithuanian", "Luxembourgish", "Macedonian", "Malagasy", "Malay", "Malayalam", "Maltese", "Maori", "Marathi", "Moldavian", "Moldovan", "Mongolian", "Myanmar", "Nepali", "Norwegian", "Nynorsk", "Occitan", "Panjabi", "Pashto", "Persian", "Polish", "Portuguese", "Punjabi", "Pushto", "Romanian", "Russian", "Sanskrit", "Serbian", "Shona", "Sindhi", "Sinhala", "Sinhalese", "Slovak", "Slovenian", "Somali", "Spanish", "Sundanese", "Swahili", "Swedish", "Tagalog", "Tajik", "Tamil", "Tatar", "Telugu", "Thai", "Tibetan", "Turkish", "Turkmen", "Ukrainian", "Urdu", "Uzbek", "Valencian", "Vietnamese", "Welsh", "Yiddish", "Yoruba"]

#@markdown Whisper model. **large-v3** = best accuracy (recommended). Smaller = faster but rougher.
use_model = "large-v3" #@param ["tiny", "base", "small", "medium", "large-v1", "large-v2", "large-v3"]

#@markdown Optional hint about the audio's topic, names, or style — nudges spelling and tone. Leave blank if unsure.
prompt = "ゆっくりとした優しい囁き声。耳元で静かに語りかけるASMR。" #@param {type:"string"}

#@markdown ---
#@markdown ### ⚙️ Advanced (fine leaving as-is)
#@markdown Precision: **float16** on GPU. Use **int8** only for CPU / low memory.
compute_type = "float16" #@param ["float16", "int8"]

#@markdown How many audio chunks to process at once. Higher = faster but uses more GPU memory. Lower this if you hit an "out of memory" error.
batch_size = 28 #@param {type:"integer"}

#@markdown Line up timestamps to each word (needed for tight subtitles). Recommended: on.
word_timestamps = True #@param {type:"boolean"}

#@markdown Max characters per subtitle line (for Japanese/Chinese, which have no spaces between words).
max_words_per_segment = 20 #@param {type:"integer"}

#@markdown *(Optional)* Paste an OpenAI API key to use their hosted Whisper instead of the free local GPU. Leave blank to stay free & local.
api_key = '' #@param {type:"string"}

# ── device ────────────────────────────────────────────────────────────────────

if api_key:
  print("Using API")
  from pydub import AudioSegment
  from pydub.silence import split_on_silence
else:
  DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
  print(f"Using {'GPU' if DEVICE == 'cuda' else 'CPU ⚠️'}")

  if DEVICE == "cuda":
    !nvidia-smi -L
  else:
    if sys_platform == 'linux':
      !lscpu | grep "Model name" | awk '{$1=$1};1'
    print("Not using GPU can result in very slow execution")
    print("Enable GPU: Runtime > Change runtime type")
    if use_model not in ['tiny', 'base', 'small']:
      print("You may want to try a smaller model (tiny, base, small)")
    compute_type = "int8"

# ── language ──────────────────────────────────────────────────────────────────

import whisper as _whisper_tok
WHISPER_LANGUAGES = [k.title() for k in _whisper_tok.tokenizer.TO_LANGUAGE_CODE.keys()]

lang_code = None
if language == "Auto-Detect":
  language = None
else:
  if language not in WHISPER_LANGUAGES:
    print(f"Language '{language}' is invalid, switching to Auto-Detect")
    language = None
  else:
    lang_code = _whisper_tok.tokenizer.TO_LANGUAGE_CODE.get(language.lower())
    print(f"\nLanguage: {language} [{lang_code}]")

# ── options dict ──────────────────────────────────────────────────────────────

options = {
    'task': task,
    'verbose': True,
    'best_of': 3,
    'beam_size': 3,
    'hotwords': prompt or None,
    'word_timestamps': word_timestamps,
    # Unchanged — best accuracy on clear speech, steps up automatically
    # only when compression_ratio_threshold detects a repetition loop
    'temperature': (0.0, 0.2, 0.4, 0.6, 0.8, 1.0),
    # Whisper's own default — skips segments where it's < 60% confident
    # there is speech, preventing hallucination into silence/noise
    'no_speech_threshold': 0.3,
    # Whisper's own default — detects repetition loops by their high
    # compressibility and triggers a retry at a higher temperature
    'compression_ratio_threshold': 2.6,
}

# ── segment splitter ──────────────────────────────────────────────────────────

def split_long_segments(segments, max_chars=20):
  """ponytail: CJK-safe subtitle splitter. Chunk by character length and
  join WITHOUT spaces. The old version did ' '.join over mora-level WhisperX
  word units, which inserted spaces between every Japanese character and
  split every 8 morae -> hundreds of 3-char garbage fragments."""
  new_segments = []
  for seg in segments:
    words = [w for w in seg.get('words', []) if 'start' in w and 'end' in w]

    if not words:
      # ponytail: no usable word timestamps -> keep the whole segment as-is
      new_segments.append(seg)
      continue

    chunk = []
    for w in words:
      chunk.append(w)
      txt = ''.join(x.get('word', x.get('text', '')) for x in chunk)
      if len(txt) >= max_chars:
        new_segments.append({
            **seg,
            'start': chunk[0]['start'],
            'end': chunk[-1]['end'],
            'text': txt,
            'words': chunk,
        })
        chunk = []

    if chunk:
      txt = ''.join(x.get('word', x.get('text', '')) for x in chunk)
      new_segments.append({
          **seg,
          'start': chunk[0]['start'],
          'end': chunk[-1]['end'],
          'text': txt,
          'words': chunk,
      })

  return new_segments

# ── load model ────────────────────────────────────────────────────────────────

if not api_key:
  MODELS_WITH_ENGLISH_VERSION = ["tiny", "base", "small", "medium"]
  _model_name = use_model
  if language == "English" and use_model in MODELS_WITH_ENGLISH_VERSION:
    _model_name = use_model + ".en"

  print(f"\nLoading {_model_name} model with WhisperX...")
  model = whisperx.load_model(
      _model_name,
      DEVICE,
      compute_type=compute_type,
      language=lang_code,
      asr_options={
        # FIX: False is WhisperX's own default and explicitly recommended
        # in their repo to reduce hallucination. Your original notebook was
        # overriding this to True via coherence_preference, which is the
        # #1 documented cause of word loops on long Japanese audio.
        # WhisperX's VAD pre-segmentation already handles context between
        # chunks, so setting this False does not hurt translation quality.
        'condition_on_previous_text': False,
        # FIX: Passing these explicitly into asr_options ensures they apply
        # at the WhisperX engine level. Without this, WhisperX may ignore
        # them even if passed to model.transcribe() later.
        'no_speech_threshold': options['no_speech_threshold'],
        'compression_ratio_threshold': options['compression_ratio_threshold'],
        'temperatures': list(options['temperature']),
        'beam_size': options['beam_size'],
        'best_of': options['best_of'],
        'hotwords': options['hotwords'],
      }
  )
  print("Model loaded.\n")

# ── API setup ─────────────────────────────────────────────────────────────────

if api_key:
  api_client = OpenAI(api_key=api_key)
  api_supported_formats = ['mp3', 'mp4', 'mpeg', 'mpga', 'm4a', 'wav', 'webm']
  api_max_bytes = 25 * 1024 * 1024
  api_transcribe = (api_client.audio.transcriptions if task == 'transcribe' else api_client.audio.translations).create
  api_model = 'whisper-1'
  api_options = {'response_format': 'verbose_json'}
  if prompt:
    api_options['prompt'] = prompt
  api_temperature = options['temperature'][0] if isinstance(options['temperature'], (tuple, list)) else options['temperature']
  if isinstance(api_temperature, (float, int)):
    api_options['temperature'] = api_temperature
  else:
    raise ValueError("Invalid temperature type")

# ── transcribe ────────────────────────────────────────────────────────────────

print("-- TRANSLATE TO ENGLISH --" if task == "translate" else "-- TRANSCRIPTION --")

results = {}

for audio_path in audio_files:
  print(f"\nProcessing: {audio_path}\n")

  if api_key:
    # ── API path ──────────────────────────────────────────────────────────────
    detect_language = not language
    if not detect_language:
      api_options['language'] = lang_code

    source_audio_name_path, source_audio_ext = os.path.splitext(audio_path)
    source_audio_ext = source_audio_ext[1:]

    if source_audio_ext in api_supported_formats:
      api_audio_path = audio_path
      api_audio_ext = source_audio_ext
    else:
      api_audio_ext = 'mp3'
      api_audio_path = f'{source_audio_name_path}.{api_audio_ext}'
      subprocess.run(['ffmpeg', '-i', audio_path, api_audio_path], check=True, capture_output=True)

    api_audio_chunks = []
    audio_bytes = os.path.getsize(api_audio_path)

    if audio_bytes >= api_max_bytes:
      audio_segment_file = AudioSegment.from_file(api_audio_path, api_audio_ext)
      min_chunks = math.ceil(audio_bytes / (api_max_bytes / 2))
      max_chunk_milliseconds = int(len(audio_segment_file) // min_chunks)

      def add_chunk(api_audio_chunk):
        api_audio_chunk_path = f"{source_audio_name_path}_{len(api_audio_chunks) + 1}.{api_audio_ext}"
        api_audio_chunk.export(api_audio_chunk_path, format=api_audio_ext)
        api_audio_chunks.append(api_audio_chunk_path)

      def raw_split(big_chunk):
        subchunks = math.ceil(len(big_chunk) / max_chunk_milliseconds)
        for subchunk_i in range(subchunks):
          chunk_start = max_chunk_milliseconds * subchunk_i
          chunk_end = min(max_chunk_milliseconds * (subchunk_i + 1), len(big_chunk))
          add_chunk(big_chunk[chunk_start:chunk_end])

      non_silent_chunks = split_on_silence(audio_segment_file, seek_step=5, min_silence_len=1250, silence_thresh=-25, keep_silence=True)
      current_chunk = non_silent_chunks[0] if non_silent_chunks else audio_segment_file
      for next_chunk in non_silent_chunks[1:]:
        if len(current_chunk) > max_chunk_milliseconds:
          raw_split(current_chunk)
          current_chunk = next_chunk
        elif len(current_chunk) + len(next_chunk) <= max_chunk_milliseconds:
          current_chunk += next_chunk
        else:
          add_chunk(current_chunk)
          current_chunk = next_chunk
      if len(current_chunk) > max_chunk_milliseconds:
        raw_split(current_chunk)
      else:
        add_chunk(current_chunk)
    else:
      api_audio_chunks.append(api_audio_path)

    result = None
    for api_audio_chunk_path in api_audio_chunks:
      with open(api_audio_chunk_path, 'rb') as api_audio_file:
        api_result = api_transcribe(model=api_model, file=api_audio_file, **api_options)
        api_result = api_result.model_dump()
      api_segments = api_result['segments']
      if result:
        last_ts = result['segments'][-1]['end'] if result['segments'] else 0
        for segment in api_segments:
          segment['start'] += last_ts
          segment['end'] += last_ts
        result['segments'].extend(api_segments)
        if 'duration' in result:
          result['duration'] += api_result.get('duration', 0)
      else:
        result = api_result
        if detect_language:
          print(f"Detected language: {result['language'].title()}\n")
      if options['verbose']:
        for segment in api_segments:
          print(f"[{format_timestamp(segment['start'])} --> {format_timestamp(segment['end'])}] {segment['text']}")

  else:
    # ── WhisperX path ─────────────────────────────────────────────────────────
    audio = whisperx.load_audio(audio_path)

    print(f"Transcribing with batch_size={batch_size}...")
    result = model.transcribe(
        audio,
        batch_size=batch_size,
        language=lang_code,
        task=task,
        print_progress=True
    )

    for seg in result['segments']:
      print(f"[{seg['start']:.1f} --> {seg['end']:.1f}] {seg['text']}")

    detected_lang = result.get('language', lang_code)
    if not lang_code:
      print(f"Detected language: {detected_lang}\n")


    # Word-level alignment
    if word_timestamps and task == 'transcribe':
      try:
        print("Aligning word timestamps...")

        # Fix CVE-2025-32434: convert jonatasgrosman Japanese model to safetensors if needed
        if detected_lang == 'ja':
          import os
          from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
          _MODEL_ID = "jonatasgrosman/wav2vec2-large-xlsr-53-japanese"
          _converted = os.path.expanduser("~/.cache/huggingface/hub/models--jonatasgrosman--wav2vec2-large-xlsr-53-japanese/snapshots/converted")
          if not os.path.isfile(os.path.join(_converted, "model.safetensors")):
            print("One-time conversion of Japanese align model to safetensors...")
            Wav2Vec2Processor.from_pretrained(_MODEL_ID).save_pretrained(_converted)
            Wav2Vec2ForCTC.from_pretrained(_MODEL_ID).save_pretrained(_converted, safe_serialization=True)
            print("Conversion done.")
          align_model_name = _converted
        else:
          align_model_name = None  # WhisperX picks the default for other languages

        model_a, metadata = whisperx.load_align_model(language_code=detected_lang, device=DEVICE, model_name=align_model_name)
        result = whisperx.align(result["segments"], model_a, metadata, audio, DEVICE, return_char_alignments=False)
        print("Alignment complete.\n")

        # Free the align model — it is reloaded per file, so without this it
        # leaks VRAM across multiple files in audio_files (and across reruns).
        free_vram(model_a, metadata)
        del model_a, metadata
      except Exception as e:
        print(f"Word alignment skipped ({e})\n")

    # Split long segments into subtitle-friendly chunks
    print("Splitting segments for subtitle formatting...")
    result['segments'] = split_long_segments(result['segments'], max_chars=max_words_per_segment)
    print(f"Total segments after split: {len(result['segments'])}\n")

    if 'language' not in result:
      result['language'] = detected_lang

    # Free per-file audio tensor + CUDA cache before the next file.
    free_vram(audio)

  # ── normalise segments ────────────────────────────────────────────────────
  for segment in result['segments']:
    segment['text'] = segment['text'].strip()

  result['text'] = '\n'.join(segment['text'] for segment in result['segments'])
  results[audio_path] = result

print(f"\n✅ Done. Processed {len(results)} file(s). Run Step 4 to save results.")

## 4️⃣ 💾 Save your results

Click **▶️** to write the transcription to files.

Your files land in the **`audio_transcription`** folder (open the 📁 Files panel on the left to download them).

Pick the format(s) in the form below:

| Format | Use it for |
|--------|-----------|
| `srt` | Subtitles (most video players & editors) — **default** |
| `vtt` | Subtitles for web video |
| `txt` | Plain text, no timestamps |
| `tsv` | Spreadsheet-friendly table |
| `json` | Full data for developers |

In [ ]:
#@title 💾 Step 4 — Save results { display-mode: "form" }
#@markdown Choose your format(s) below, then click **▶️**. Files are saved to the `audio_transcription` folder.

output_dir = "audio_transcription"

#@markdown **Output format(s)** — comma-separate to save several at once (e.g. `txt,srt`):
output_formats = "srt" #@param ["txt,vtt,srt,tsv,json", "txt,vtt,srt", "txt,vtt", "txt,srt", "txt", "vtt", "srt", "tsv", "json"] {allow-input: true}
output_formats = output_formats.split(',')

import re
import copy
import glob
from typing import TextIO

def clean_repeated_words(text, threshold=4):
  """
  Remove hallucinated word repetitions from Whisper output.
  e.g. 'more more more more more' → 'more'
  e.g. 'I can't do that... I can't do that...' → 'I can't do that...'
  threshold: how many repeats before we consider it a hallucination
  """
  if not text:
    return text

  # ponytail: Step 0 - collapse no-space CJK repeats (ぐりぐりぐり... -> ぐり,
  # すりすり... -> すり). The space-based steps below never match Japanese
  # because there are no spaces between the repeats.
  text = re.sub(r'(.{1,10}?)\1{3,}', r'\1', text)

  # Step 1: Collapse repeated single words
  # e.g. "more more more more" → "more"
  text = re.sub(
    r'\b(\w+)(\s+\1){' + str(threshold - 1) + r',}\b',
    r'\1',
    text,
    flags=re.IGNORECASE
  )

  # Step 2: Collapse repeated short phrases (2–6 words)
  # e.g. "I can't do that I can't do that I can't do that" → "I can't do that"
  text = re.sub(
    r'(.{8,60}?)(\s+\1){' + str(threshold - 2) + r',}',
    r'\1',
    text,
    flags=re.IGNORECASE
  )

  # Step 3: Clean up leftover punctuation artifacts from collapsing
  # e.g. "that... that... that..." → "that..."
  text = re.sub(r'(\.\.\.)(\s*\.\.\.)+', r'\1', text)
  text = re.sub(r'\s{2,}', ' ', text).strip()

  return text

def clean_segments(segments):
  """Apply repetition cleaning to all segment texts."""
  cleaned = []
  seen_texts = []

  for seg in segments:
    text = seg.get('text', '').strip()

    # Clean within-segment repetitions
    text = clean_repeated_words(text)

    # Drop segment entirely if it's an exact or near-duplicate of
    # the previous 2 segments (cross-segment hallucination)
    is_duplicate = False
    for prev in seen_texts[-2:]:
      if text and prev and (
        text == prev or
        (len(text) > 10 and text in prev) or
        (len(prev) > 10 and prev in text)
      ):
        is_duplicate = True
        break

    if not is_duplicate and text:
      seen_texts.append(text)
      cleaned.append({**seg, 'text': text})

  return cleaned

class WriteText(WriteTXT):
  def write_result(self, result: dict, file: TextIO, **kwargs):
    print(result['text'], file=file, flush=True)

def write_result(result, output_format, output_file_name):
  output_format = output_format.strip()

  safe_result = copy.deepcopy(result)

  # Clean repeated/hallucinated words from all segments
  safe_result['segments'] = clean_segments(safe_result['segments'])

  # Rebuild full text from cleaned segments
  safe_result['text'] = '\n'.join(s['text'] for s in safe_result['segments'])

  # Strip 'words' to avoid KeyError: 'start' in whisper writer
  for seg in safe_result.get('segments', []):
    seg.pop('words', None)

  fix_vtt = output_format == 'vtt' and safe_result['segments'] and safe_result['segments'][0].get('start') == 0
  if fix_vtt:
    safe_result['segments'][0]['start'] += 1/1000

  writer = WriteText(output_dir) if output_format == 'txt' else get_writer(output_format, output_dir)

  writer_options = {
    'max_line_width': None,
    'max_line_count': None,
    'highlight_words': False,
  }
  writer(safe_result, output_file_name, writer_options)

  if fix_vtt:
    safe_result['segments'][0]['start'] = 0

  output_file_path = os.path.join(output_dir, f"{output_file_name}.{output_format}")
  print(output_file_path)

print("Writing results...")
os.makedirs(output_dir, exist_ok=True)

for audio_path, result in results.items():
  print(end='\n')
  output_file_name = os.path.splitext(os.path.basename(audio_path))[0]
  for output_format in output_formats:
    write_result(result, output_format, output_file_name)

# ── Post-process SRT/VTT: UTF-8 BOM + CRLF line endings ───────────────────
print("\nFixing subtitle file encoding...")
for ext in ['srt', 'vtt']:
  for filepath in glob.glob(os.path.join(output_dir, '*.{}'.format(ext))):
    with open(filepath, 'rb') as f:
      raw = f.read()

    if raw.startswith(b'\xef\xbb\xbf'):
      raw = raw[3:]

    raw = raw.replace(b'\r\n', b'\n').replace(b'\r', b'\n')
    raw = raw.replace(b'\n', b'\r\n')

    with open(filepath, 'wb') as f:
      f.write(b'\xef\xbb\xbf')
      f.write(raw)

    print(f"✅ Fixed: {filepath}")

print("\n✅ All done. Your files are in the 'audio_transcription' folder (📁 Files panel on the left).")

## 5️⃣ 💬 Translate with DeepL *(optional)*

Want the transcript in another language, with nicer wording than Whisper's built-in translate? Run this step to translate it with **DeepL**.

**Setup (one time):**
1. Get a free key at [deepl.com/pro-api](https://www.deepl.com/pro-api).
2. In Colab, open the **🔑 Secrets** panel (left sidebar) → **+ Add new secret**.
3. Name it **`DEEPL_API_KEY`**, paste your key, and turn **Notebook access ON**.

Then pick your target language in the form and click **▶️**. Translated files are saved next to the originals.

> Skip this step if you don't need a DeepL translation.

In [ ]:
#@title 💬 Step 5 — Translate with DeepL { display-mode: "form" }
#@markdown Add your `DEEPL_API_KEY` in the 🔑 Secrets panel first (see the notes above), then set the options and click **▶️**.

import os
import deepl
import whisper as _whisper_tok
import time

# Read the DeepL key from the environment, or fall back to the Colab Secret
# named DEEPL_API_KEY (🔑 panel on the left, with "Notebook access" ON).
deepl_api_key = os.environ.get("DEEPL_API_KEY", "")
if not deepl_api_key:
  try:
    from google.colab import userdata
    deepl_api_key = userdata.get('DEEPL_API_KEY') or ""
  except Exception:
    deepl_api_key = ""

#@markdown **Translate into:**
deepl_target_language = "English (American)" #@param ["", "Bulgarian", "Chinese (simplified)", "Czech", "Danish", "Dutch", "English (American)", "English (British)", "Estonian", "Finnish", "French", "German", "Greek", "Hungarian", "Indonesian", "Italian", "Japanese", "Korean", "Latvian", "Lithuanian", "Norwegian", "Polish", "Portuguese (Brazilian)", "Portuguese (European)", "Romanian", "Russian", "Slovak", "Slovenian", "Spanish", "Swedish", "Turkish", "Ukrainian"]
#@markdown **Tone** — formal vs informal (only some languages support this):
deepl_formality = "default" #@param ["default", "formal", "informal"]
#@markdown **Context** — "Share context" reads neighbouring lines for smoother, more consistent translations:
deepl_coherence_preference = "Share context between lines" #@param ["Share context between lines", "Translate each line independently"]
deepl_coherence_preference = deepl_coherence_preference == "Share context between lines"

if not deepl_api_key:
  print("Required: deepl_api_key")
  print("Get a DeepL Developer Account API Key: https://www.deepl.com/pro-api")

if not deepl_target_language:
  print("Required: deepl_target_language")
elif deepl_target_language == 'English':
  deepl_target_language = "English (British)"
elif deepl_target_language == 'Chinese':
  deepl_target_language = "Chinese (simplified)"
elif deepl_target_language == 'Portuguese':
  deepl_target_language = "Portuguese (European)"

use_deepl_translation = deepl_api_key and deepl_target_language

# FIX: retry wrapper — deepl 1.27.0 hardcodes a 10s timeout internally with
# no constructor parameter to change it. Retrying with backoff is the only
# reliable solution for the free tier which is slower than paid.
def translate_with_retry(translator, max_retries=5, **kwargs):
  for attempt in range(max_retries):
    try:
      return translator.translate_text(**kwargs)
    except Exception as e:
      if 'timed out' in str(e).lower() and attempt < max_retries - 1:
        wait = 2 ** attempt  # 1s, 2s, 4s, 8s...
        print(f"  ⚠️ Timeout on attempt {attempt + 1}/{max_retries}, retrying in {wait}s...")
        time.sleep(wait)
      else:
        raise

if use_deepl_translation:
  if deepl_formality != 'default':
    deepl_formality = 'prefer_more' if deepl_formality == 'formal' else 'prefer_less'

  translated_results = {}

  try:
    deepl_translator = deepl.Translator(deepl_api_key)
    deepl_source_languages = [lang.code.upper() for lang in deepl_translator.get_source_languages()]
    deepl_target_languages_dict = deepl_translator.get_target_languages()
    deepl_target_languages = [lang.name for lang in deepl_target_languages_dict]
    deepl_target_language_code = next(lang.code for lang in deepl_target_languages_dict if lang.name == deepl_target_language).upper()
    target_language_code = deepl_target_language_code.split('-')[0]

    for audio_path, result in results.items():
      deepl_usage = deepl_translator.get_usage()
      if deepl_usage.any_limit_reached:
        print(audio_path)
        raise deepl.DeepLException("Quota for this billing period has been exceeded, message: Quota Exceeded")
      else:
        print(audio_path + '\n')

      source_language_code = _whisper_tok.tokenizer.TO_LANGUAGE_CODE.get(result['language'].lower(), '').upper()

      # Fix: force correct Japanese code
      if result['language'].lower() in ('japanese', 'ja'):
        source_language_code = 'JA'

      if (task == 'translate' and target_language_code != 'EN') or (task == 'transcribe' and source_language_code in deepl_source_languages and source_language_code != target_language_code):
        source_lang = source_language_code if task == 'transcribe' else None
        translate_from = f"from {result['language'].title()} [{source_language_code}] " if source_lang else ''
        print(f"DeepL: Translate results {translate_from}to {deepl_target_language} [{deepl_target_language_code}]\n")

        segments = result['segments']
        translated_results[audio_path] = {'text': '', 'segments': [], 'language': deepl_target_language}

        # Reduced from 200 — smaller batches finish faster, less likely to timeout
        deepl_batch_requests_size = 50

        total_batches = math.ceil(len(segments) / deepl_batch_requests_size)
        for batch_num, batch_segments in enumerate([segments[i:i + deepl_batch_requests_size] for i in range(0, len(segments), deepl_batch_requests_size)]):
          print(f"  Translating batch {batch_num + 1}/{total_batches}...")
          batch_segments_text = [segment['text'] for segment in batch_segments]
          if deepl_coherence_preference:
            batch_segments_text = '<br/>'.join(batch_segments_text)

          # Use retry wrapper instead of direct call
          deepl_results = translate_with_retry(
              deepl_translator,
              text=batch_segments_text,
              source_lang=source_lang,
              target_lang=deepl_target_language_code,
              formality=deepl_formality,
              split_sentences='nonewlines',
              tag_handling='xml' if deepl_coherence_preference else None,
              ignore_tags='br' if deepl_coherence_preference else None,
              outline_detection=False if deepl_coherence_preference else None
          )

          deepl_results_segments = deepl_results.text.split('<br/>') if deepl_coherence_preference else [r.text for r in deepl_results]

          for j, translated_text in enumerate(deepl_results_segments):
            segment = batch_segments[j]
            translated_text = translated_text.lstrip(',.。 ').rstrip()
            if not deepl_coherence_preference and translated_text and translated_text[-1] in '.。' and segment['text'][-1] not in '.。':
              translated_text = translated_text[:-1]
            translated_results[audio_path]['segments'].append(dict(id=segment.get('id', j), start=segment['start'], end=segment['end'], text=translated_text))
            if options['verbose']:
              print(f"[{format_timestamp(segment['start'])} --> {format_timestamp(segment['end'])}] {translated_text}")

          # Pause between batches to avoid free tier rate limiting
          time.sleep(1)

        deepl_usage = deepl_translator.get_usage()
        if deepl_usage.character.valid:
          print(f"\nDeepL: Character usage: {deepl_usage.character.count} / {deepl_usage.character.limit} ({100*(deepl_usage.character.count/deepl_usage.character.limit):.2f}%)\n")
      elif source_language_code == target_language_code:
        print(f"Nothing to translate. Results are already in {result['language']}.")
      elif task == 'transcribe' and source_language_code not in deepl_source_languages:
        print(f"DeepL: {result['language']} is not yet supported")
  except deepl.DeepLException as e:
    if isinstance(e, deepl.AuthorizationException) and str(e) == "Authorization failure, check auth_key":
      e = "Authorization failure, check deepl_api_key"
    print(f"\nDeepL: [Error] {e}\n")

  if translated_results:
    print("Writing translated results...")
    for audio_path, translated_result in translated_results.items():
      print(end='\n')
      translated_result['text'] = '\n'.join(s['text'] for s in translated_result['segments'])
      output_file_name = os.path.splitext(os.path.basename(audio_path))[0]
      translated_output_file_name = f"{output_file_name}_{deepl_target_language}"
      for output_format in output_formats:
        write_result(translated_result, output_format, translated_output_file_name)